# M2 · Mini-projet — Équilibrer le trafic d'un quartier

| Module | Durée | Prérequis |
|---|---|---|
| M2 – Systèmes linéaires | 2 h 30 | M0, notebooks 2.1 à 2.3 |

In [ ]:
# Style graphique Liouaï : couleurs du logo, axes sobres
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

LIOUAI = {"navy": "#243743", "vert": "#28B78D", "vert_texte": "#167A5C", "gris": "#A1A9AE",
          "gris_texte": "#5C6870", "jaune": "#E5C800", "fond": "#F3F5F6"}
plt.rcParams.update({
    "axes.prop_cycle": plt.cycler(color=[LIOUAI["vert"], LIOUAI["navy"], LIOUAI["gris"],
                                         LIOUAI["jaune"], LIOUAI["vert_texte"], LIOUAI["gris_texte"]]),
    "axes.edgecolor": LIOUAI["gris"], "axes.labelcolor": LIOUAI["navy"],
    "axes.titlecolor": LIOUAI["navy"], "axes.titleweight": "bold",
    "axes.spines.top": False, "axes.spines.right": False,
    "xtick.color": LIOUAI["gris_texte"], "ytick.color": LIOUAI["gris_texte"],
    "text.color": LIOUAI["navy"], "grid.color": "#E1E6E9", "legend.frameon": False,
    "font.family": "sans-serif", "font.sans-serif": ["Roboto", "Helvetica", "Arial", "DejaVu Sans"],
    "figure.dpi": 100,
})
# Dégradé de marque : gris clair -> vert -> navy (et sa version inversée)
if "liouai" not in mpl.colormaps:
    _cmap = LinearSegmentedColormap.from_list("liouai", [LIOUAI["fond"], LIOUAI["vert"], LIOUAI["navy"]])
    mpl.colormaps.register(_cmap)
    mpl.colormaps.register(_cmap.reversed())

## ① Définir la question

La mairie veut installer des capteurs de comptage dans un petit quartier. Les quatre rues intérieures forment un carré à sens unique : **A → B → C → D → A**. Aux quatre carrefours, des rues extérieures font entrer et sortir des voitures. On connaît les flux extérieurs moyens à l'heure de pointe (voitures par heure) :

| Carrefour | Entrée extérieure | Sortie extérieure |
|---|---|---|
| A (nord-ouest) | 400 | 250 |
| B (nord-est) | 300 | 500 |
| C (sud-est) | 600 | 300 |
| D (sud-ouest) | 200 | 450 |

On ne connaît **pas** les flux dans les quatre rues intérieures : $x_1$ (A→B), $x_2$ (B→C), $x_3$ (C→D), $x_4$ (D→A).

**Activité** : Dans la cellule suivante, répondez en quelques lignes :
1. Quelles informations du texte sont utiles ? Lesquelles manquent ?
2. Formulez une question précise à laquelle le calcul doit répondre.

Indiquez vos réponses et justifiez-les.

**Hypothèses.** Complétez la liste. Pour chacune, indiquez pourquoi on la fait.

- H1 — Conservation : à chaque carrefour, il entre autant de voitures qu'il en sort (pas de parking, pas d'accident). *Pourquoi ?* …
- H2 — Les flux sont des moyennes stables sur l'heure considérée. *Pourquoi ?* …
- H3 — …

## ② Estimer avant de calculer

Avant tout calcul, pariez. Le notebook comparera vos paris au résultat à la fin.

In [ ]:
# Vos paris (modifiez les valeurs)
pari_nombre_de_solutions = "une"      # "aucune", "une" ou "une infinité"
pari_flux_min_x1 = 300                # flux minimal plausible dans la rue A→B (voitures/h)

## ③ Abstraire

**Schéma.** Dessinez le carré A-B-C-D avec les flèches intérieures et extérieures. Le schéma ci-dessous sert de vérification.

![Schéma du quartier : quatre carrefours A, B, C, D reliés par des rues à sens unique, avec les flux extérieurs connus](../../assets/schemas/schema-quartier.svg)

**Mise en équations.** H1 donne une équation par carrefour : *entrées = sorties*.

$$
\begin{cases}
x_4 + 400 = x_1 + 250 & \text{(carrefour A)} \\
x_1 + 300 = x_2 + 500 & \text{(carrefour B)} \\
x_2 + 600 = x_3 + 300 & \text{(carrefour C)} \\
x_3 + 200 = x_4 + 450 & \text{(carrefour D)}
\end{cases}
$$

**Activité** : Réécrivez chaque équation sous la forme « inconnues à gauche, constante à droite », puis vérifiez la matrice `M` et le vecteur `b` ci-dessous.

**Choix de l'outil.** Trois outils possibles : `numpy.linalg.solve` (numérique, système carré inversible), `sympy` (calcul exact, forme échelonnée), `numpy.linalg.lstsq` (moindres carrés). Lequel choisiriez-vous a priori, et pourquoi ?

Indiquez vos réponses et justifiez-les.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Une ligne par carrefour (A, B, C, D), une colonne par inconnue (x1, x2, x3, x4)
# Convention : (flux intérieur sortant) - (flux intérieur entrant) = (entrée extérieure) - (sortie extérieure)
M = np.array([
    [ 1,  0,  0, -1],   # A : x1 - x4 =  150
    [-1,  1,  0,  0],   # B : x2 - x1 = -200
    [ 0, -1,  1,  0],   # C : x3 - x2 =  300
    [ 0,  0, -1,  1],   # D : x4 - x3 = -250
], dtype=float)
b = np.array([150, -200, 300, -250], dtype=float)
M, b

## ④ Calculer

### Premier essai : l'outil le plus direct

In [ ]:
try:
    x = np.linalg.solve(M, b)
    print("Solution :", x)
except np.linalg.LinAlgError as err:
    print("numpy refuse de résoudre :", err)

**Isoler la cause.** L'outil a échoué. Avant de changer d'outil, comprenons pourquoi. Calculez le rang de `M` et celui de la matrice augmentée `[M | b]`.

In [ ]:
rang_M = np.linalg.matrix_rank(M)
rang_aug = np.linalg.matrix_rank(np.column_stack([M, b]))
print(f"rang(M) = {rang_M}, rang([M|b]) = {rang_aug}, nombre d'inconnues = {M.shape[1]}")
print("Somme des 4 lignes de M :", M.sum(axis=0), "  somme de b :", b.sum())

**Activité** : Que signifie le fait que la somme des quatre équations donne $0 = 0$ ? Traduisez-le dans le contexte : que vaut le total des voitures entrant dans le quartier comparé au total sortant ?

Indiquez vos réponses et justifiez-les.

### Deuxième outil : le calcul exact avec SymPy

In [ ]:
import sympy as sp

x1, x2, x3, x4 = sp.symbols("x1 x2 x3 x4")
Ms, bs = sp.Matrix(M.astype(int)), sp.Matrix(b.astype(int))
echelonnee, pivots = Ms.row_join(bs).rref()
display(echelonnee)
print("Colonnes pivots :", pivots)

solution = sp.linsolve((Ms, bs), x1, x2, x3, x4)
solution

### Troisième outil : décrire toutes les solutions avec NumPy/SciPy

Toute solution s'écrit comme *une solution particulière plus un vecteur du noyau de $M$* :

$$
\mathbf{x} = \mathbf{x}_p + t\,\mathbf{n}, \qquad M\mathbf{n} = \mathbf{0}, \qquad t \in \mathbb{R}.
$$

In [ ]:
from scipy.linalg import null_space

x_part, *_ = np.linalg.lstsq(M, b, rcond=None)   # une solution particulière (celle de norme minimale)
noyau = null_space(M)[:, 0]
noyau = noyau / noyau[0]                           # on normalise pour que x1 avance de 1
print("Solution particulière :", np.round(x_part, 2))
print("Direction du noyau    :", np.round(noyau, 2))
print("Vérification M @ x_part - b =", np.round(M @ x_part - b, 10))

## ⑤ Interpréter

Écrivons les solutions en fonction du flux $t = x_1$ dans la rue A→B.

In [ ]:
t = np.linspace(0, 600, 601)
flux = np.array([x_part + (ti - x_part[0]) * noyau for ti in t])   # chaque ligne = (x1, x2, x3, x4)

fig, ax = plt.subplots(figsize=(7, 4))
for i, nom in enumerate(["x₁ (A→B)", "x₂ (B→C)", "x₃ (C→D)", "x₄ (D→A)"]):
    ax.plot(t, flux[:, i], label=nom)
ax.axhline(0, color=LIOUAI["navy"], lw=0.8)
admissible = (flux >= 0).all(axis=1)
t_min = t[admissible].min()
ax.axvspan(t_min, t.max(), color=LIOUAI["vert"], alpha=0.12, label="tous les flux ≥ 0")
ax.set_xlabel("t = flux dans la rue A→B (voitures/h)")
ax.set_ylabel("flux (voitures/h)")
ax.legend(); ax.set_title("Toutes les répartitions compatibles avec les comptages extérieurs")
plt.show()
print(f"Flux minimal admissible dans A→B : {t_min:.0f} voitures/h")

**Activité** :
1. Combien de solutions mathématiques le système possède-t-il ? Combien sont physiquement possibles ?
2. Pourquoi $x_1$ ne peut-il pas descendre sous la valeur trouvée ? Quelle rue serait alors « en sens interdit » ?
3. Que répondre à la mairie : combien de capteurs intérieurs faut-il au minimum pour connaître tous les flux ?

Indiquez vos réponses et justifiez-les.

In [ ]:
# Comparaison avec vos paris
print(f"Nombre de solutions — pari : {pari_nombre_de_solutions!r} ; réalité : une infinité (rang {rang_M} < 4 inconnues)")
print(f"Flux minimal dans A→B — pari : {pari_flux_min_x1} ; réalité : {t_min:.0f} "
      f"(écart : {pari_flux_min_x1 - t_min:+.0f} voitures/h)")

### Avec un capteur

La mairie installe un seul capteur, dans la rue A→B, et mesure **350 voitures/h**. On ajoute cette équation au système.

In [ ]:
M2 = np.vstack([M, [1, 0, 0, 0]])
b2 = np.append(b, 350)
x_capteur, residu, rang2, _ = np.linalg.lstsq(M2, b2, rcond=None)
print("rang du nouveau système :", rang2)
print("Flux intérieurs (voitures/h) :", dict(zip(["x1", "x2", "x3", "x4"], [int(v) for v in np.round(x_capteur)])))

## ⑥ Critiquer et généraliser

**Hypothèse la plus fragile.** Classez H1, H2 et vos hypothèses de la plus fragile à la plus solide. Pour la plus fragile, que se passe-t-il si elle est fausse ? (Indice : modifiez une sortie extérieure de 10 voitures et relancez le calcul du rang.)

**Vérifier autrement.** Proposez une seconde méthode pour contrôler le résultat avec capteur. (Par exemple : remplacer les valeurs dans chaque équation de carrefour.)

In [ ]:
# Vérification équation par équation
noms = ["A", "B", "C", "D", "capteur"]
for nom, ligne, cible in zip(noms, M2, b2):
    print(f"{nom:8s} : {ligne @ x_capteur:8.1f}  attendu {cible:8.1f}")

**Généraliser.** Transformez les flux extérieurs en paramètres : écrivez une fonction `flux_interieurs(entrees, sorties, mesure_x1)` qui renvoie les quatre flux, ou `None` si les comptages extérieurs sont incohérents. Testez-la sur un cas où le total entrant diffère du total sortant.

In [ ]:
def flux_interieurs(entrees, sorties, mesure_x1):
    # À compléter
    ...

# Tests
# print(flux_interieurs([400, 300, 600, 200], [250, 500, 300, 450], 350))   # attendu : [350, 150, 450, 200]
# print(flux_interieurs([400, 300, 600, 200], [250, 500, 300, 460], 350))   # attendu : None

**Pour aller plus loin.** Que devient le problème pour un quartier de 3 × 3 carrefours ? Le nombre de capteurs nécessaires est-il lié au nombre de « boucles » du réseau ?

## Bilan — auto-évaluation

Pour chaque compétence travaillée dans ce notebook, choisissez un niveau : **0** non observé · **1** guidé · **2** autonome · **3** transfert.
La cellule enregistre votre bilan dans `progression.json`, à la racine du site.

In [ ]:
import json, datetime, pathlib

bilan = {
    "Identifier, énoncer et justifier les hypothèses": 1,
    "Formuler des questions précises": 1,
    "Faire des schémas pour structurer les connaissances": 1,
    "Avoir l'intuition du comportement de l'outil": 1,
    "Isoler la cause d'un problème de fonctionnement": 1,
    "Énoncer le résultat dans le contexte du problème": 1,
    "Quantifier la validité et l'impact des hypothèses": 1,
    "Transformer des constantes en paramètres": 1,
}   # modifiez les niveaux

ici = pathlib.Path.cwd()
racine = next((p for p in [ici, *ici.parents] if (p / "myst.yml").exists()), ici)
fichier = racine / "progression.json"
historique = json.loads(fichier.read_text(encoding="utf-8")) if fichier.exists() else []
historique.append({"notebook": "M2-projet-trafic", "date": datetime.date.today().isoformat(), "niveaux": bilan})
fichier.write_text(json.dumps(historique, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"Bilan enregistré ({len(historique)} entrée(s) dans {fichier.name}).")